# Construct Data

This step is used for create derived attributes, or entire new records, or transforming values from existing attributes.

In [1]:
import pandas as pd
from pathlib import Path
from omegaconf import OmegaConf
from olist_ml.config import OlistConfig
from sqlalchemy import create_engine

root_path = project_root=Path.cwd().parent.parent
cfg = OmegaConf.load("../../conf/config.yaml")

raw_data_path = root_path / cfg.paths.raw_data
olist_sqlite_path = raw_data_path / cfg.files.olist_sqlite
db_file = f"sqlite:///{olist_sqlite_path}"
engine = create_engine(db_file)

In [3]:
demand_by_category = pd.read_sql("""
    SELECT
        strftime('%Y-%m', o.order_purchase_timestamp) as order_purchase_date,
        pcn.product_category_name_english as category_name,
        SUM(oi.price + oi.freight_value) as total_price
    FROM order_items oi
    LEFT JOIN orders o ON o.order_id = oi.order_id
    LEFT JOIN products p ON oi.product_id = p.product_id
    LEFT JOIN product_category_name_translation pcn ON p.product_category_name = pcn.product_category_name
    WHERE p.product_category_name IN (
        SELECT
            p2.product_category_name
        FROM order_items oi2
        LEFT JOIN products p2 ON oi2.product_id = p2.product_id
        WHERE oi2.price <= 8000 
          AND oi2.freight_value <= 80
        GROUP BY p2.product_category_name
        HAVING COUNT(*) >= 3000
    )
    GROUP BY category_name, order_purchase_date
    ORDER BY order_purchase_date
""", engine)

In [5]:
demand_by_category.head(10)

,order_purchase_date,category_name,total_price
0,2016-09,furniture_decor,136.23
1,2016-09,health_beauty,143.46
2,2016-09,telephony,75.06
3,2016-10,auto,2257.56
4,2016-10,bed_bath_table,606.58
5,2016-10,computers_accessories,1740.09
6,2016-10,cool_stuff,1327.11
7,2016-10,furniture_decor,7052.28
8,2016-10,garden_tools,1465.19
9,2016-10,health_beauty,5493.38
